In [89]:
#data initializng
import pandas as pd
df = pd.read_csv("AmesHousing.csv")
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [90]:
#data cleaning and feature selection
#df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   object 
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   object 
 7   Alley            198 non-null    object 
 8   Lot Shape        2930 non-null   object 
 9   Land Contour     2930 non-null   object 
 10  Utilities        2930 non-null   object 
 11  Lot Config       2930 non-null   object 
 12  Land Slope       2930 non-null   object 
 13  Neighborhood     2930 non-null   object 
 14  Condition 1      2930 non-null   object 
 15  Condition 2      2930 non-null   object 
 16  Bldg Type        2930 non-null   object 
 17  House Style   

In [91]:
#dropping useless features
df = df.drop("Order", axis=1)
df = df.drop("PID", axis=1)
df.duplicated().sum() #0

np.int64(0)

In [92]:
#handling missing values and converting datatypes
# df.isna().sum()
categ_cols = df.select_dtypes(include="object").columns
df[categ_cols] = df[categ_cols].fillna(df[categ_cols].mode().iloc[0])
df = pd.get_dummies(df, columns=categ_cols,dtype="int")
num_col = df.select_dtypes(include="number").columns
df[num_col] = df[num_col].fillna(df[num_col].mean())



In [87]:
df.isna().sum()[df.isna().sum() > 0]

Series([], dtype: int64)

In [93]:
# Model creation
import math
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

X = df.drop("SalePrice", axis=1)
Y = df["SalePrice"]

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=43
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, Y_train)
Y_pred = model.predict(X_test_scaled)
mse = mean_squared_error(Y_test, Y_pred)
rmse = math.sqrt(mse)
r2 = r2_score(Y_test, Y_pred)
scores = cross_val_score(
    model,
    X_train_scaled,
    Y_train,
    cv=6,
    scoring="neg_mean_squared_error"
)

rmse_scores = np.sqrt(-scores)

print(f"r2: {r2}")
print(f"rmse: {rmse}")
print(f"mse: {mse}")
print(f"len of test input: {len(X_test)}")
print(f"cross_val_score: {rmse_scores.mean()}")

r2: 0.815836003186643
rmse: 30617.47233802183
mse: 937429612.369532
len of test input: 586
cross_val_score: 28870.582484868693
